In [ ]:
import sdmx
import pandas as pd
import time
from pathlib import Path
from tenacity import retry, stop_after_attempt, wait_random_exponential


# ===========================
# CONFIGURATION
# ===========================

DATAFLOW_ID = "PRC_HICP_MINR"
OUTPUT_DIR = Path("src", "assets", "data")
MAPS_DIR = Path("src", "assets", "maps")
OUTPUT_DIR.mkdir(exist_ok=True)

FREQ = "M"
START_PERIOD = "2000-01"

# ===========================
# SDMX CLIENT
# ===========================

client = sdmx.Client("ESTAT")  # Eurostat shortcut name



In [2]:
# ===========================
# LOAD DATA MAPS
# ===========================

code_map = pd.read_csv(MAPS_DIR / "coicop18.csv")

In [ ]:
# ===========================
# GET STRUCTURE
# ===========================

print("Downloading data structure...")

try:
    dsd_response = client.get(
        resource_type="datastructure",
        resource_id=DATAFLOW_ID
    )
except Exception as e:
    raise RuntimeError(f"Failed to download DSD: {e}")

In [4]:
dsd = dsd_response.structure[DATAFLOW_ID]

dimensions = list(dsd.dimensions.components)

In [5]:
@retry(stop=stop_after_attempt(7),wait= wait_random_exponential(multiplier=1, max=60), reraise=True)
def download_data(item_id, dimensions, FREQ, START_PERIOD, DATAFLOW_ID, OUTPUT_DIR):
    """Download data for a specific ECOICOPv2 item and save to CSV."""
    
    geo_key = "EU+EA+BE+BG+CZ+DK+DE+EE+IE+EL+ES+FR+HR+IT+CY+LV+LT+LU+HU+MT+NL+AT+PL+PT+RO+SI+SK+FI+SE"
    unit_key = "I25+RCH_A+RCH_M"
    # Detect dimensions
    freq_dim = next(d for d in dimensions if d.id.lower() == "freq")
    geo_dim = next(d for d in dimensions if d.id.lower() == "geo")
    item_dim = next(d for d in dimensions if "coicop" in d.id.lower())
    unit_dim = next((d for d in dimensions if "unit" in d.id.lower()), None)  # Optional

    key = {
        freq_dim.id: FREQ,
        item_dim.id: item_id,
        geo_dim.id: geo_key,
        unit_dim.id: unit_key
    }

    response = client.get(
        resource_type="data",
        resource_id=DATAFLOW_ID,
        key=key,
        params={
            "startPeriod": START_PERIOD
        }
    )

    data = response.data

    if data is None:
        print("No data found.")
        return False

    df = sdmx.to_pandas(data)

    if df.empty:
        print("Empty dataset.")
        return False

    df = df.reset_index()

    output_file = OUTPUT_DIR / f"{item_id}.csv"
    df.to_csv(output_file, index=False)
    print(f"Saved {output_file}")

    time.sleep(0.3)
    

In [6]:
# ===========================
# DOWNLOAD LOOP
# ===========================

for item_id in code_map["code"]:
    print(f"\nProcessing item: {item_id}")
    
    try:
        download_data(item_id, dimensions, FREQ, START_PERIOD, DATAFLOW_ID, OUTPUT_DIR)

    except Exception as e:
        print(f"Error for item {item_id}: {e}")
        continue

print("\nDone.")


Processing item: TOTAL
Saved assets/data/TOTAL.csv

Processing item: CP01
Saved assets/data/CP01.csv

Processing item: CP011
Saved assets/data/CP011.csv

Processing item: CP0111
Saved assets/data/CP0111.csv

Processing item: CP01111
Saved assets/data/CP01111.csv

Processing item: CP01112
Saved assets/data/CP01112.csv

Processing item: CP01113
Saved assets/data/CP01113.csv

Processing item: CP011131
Saved assets/data/CP011131.csv

Processing item: CP011139
Saved assets/data/CP011139.csv

Processing item: CP01114
Saved assets/data/CP01114.csv

Processing item: CP01115
Saved assets/data/CP01115.csv

Processing item: CP01119
Saved assets/data/CP01119.csv

Processing item: CP0112
Saved assets/data/CP0112.csv

Processing item: CP01121
Saved assets/data/CP01121.csv

Processing item: CP01122
Saved assets/data/CP01122.csv

Processing item: CP011221
Saved assets/data/CP011221.csv

Processing item: CP011222
Saved assets/data/CP011222.csv

Processing item: CP011223
Saved assets/data/CP011223.csv


In [5]:


# Detect dimensions
freq_dim = next(d for d in dimensions if d.id.lower() == "freq")
geo_dim = next(d for d in dimensions if d.id.lower() == "geo")
item_dim = next(d for d in dimensions if "coicop" in d.id.lower())
unit_dim = next((d for d in dimensions if "unit" in d.id.lower()), None)  # Optional

print("Dimensions detected:", freq_dim.id, geo_dim.id, item_dim.id, unit_dim.id if unit_dim else "None")



Dimensions detected: freq geo coicop18 unit


In [6]:
geo_key = "EU+EA+BE+BG+CZ+DK+DE+EE+IE+EL+ES+FR+HR+IT+CY+LV+LT+LU+HU+MT+NL+AT+PL+PT+RO+SI+SK+FI+SE"

In [7]:
unit_key = "I25+RCH_A+RCH_M"

In [ ]:
# Get code lists
geo_codes = geo_dim.local_representation.enumerated
item_codes = item_dim.local_representation.enumerated

In [15]:
sdmx.to_pandas(item_codes.items).to_csv("ecoicop_codes.csv", index=True, header=False)